# Notebook 02: Remote Mistral-7B Inference Service on Google Colab GPU

**Project**: Multi-Context Intelligent RAG System (Customer Support Capstone)  
**Environment**: Google Colab (Free NVIDIA T4 GPU)  
**Storage Root**: Google Drive (`/content/drive/My Drive/Internship_RAG_MCP_Capstone`)  
**Primary Model**: `mistralai/Mistral-7B-Instruct-v0.3` (Memory-efficient 4-bit NF4 quantized via `bitsandbytes`)  
**Target Endpoint**: OpenAI-compatible REST server on port `8001` exposed via authenticated Cloudflare / ngrok tunnel

---

### Architectural Governance
1. **Zero Local Storage Overhead**: Model weights (~4.5 GB quantized) reside solely in Colab ephemeral disk; **0 bytes downloaded to local Windows computer**.
2. **Authenticated Endpoint**: Requires an authorization token header (`Authorization: Bearer <INFERENCE_AUTH_TOKEN>`) to prevent unauthorized access.
3. **Bounded Compute**: Enforces bounded token limits (`max_tokens=512`, `temperature=0.2`) and 30-second request timeouts.
4. **Safe Resource Sizing**: Verified for standard Google Colab T4 (15 GB VRAM) using 4-bit quantized weights (~5.2 GB VRAM footprint).

## Step 1: Mount Google Drive & Verify Persistence Hierarchy

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/My Drive/Internship_RAG_MCP_Capstone')
    print(f"[SUCCESS] Google Drive mounted. Project root exists: {DRIVE_ROOT.exists()}")
    
    # Verify knowledge base and backups
    kb_dir = DRIVE_ROOT / "03_Knowledge_Base"
    backup_dir = DRIVE_ROOT / "11_Backups"
    print(f"  + Knowledge Base: {kb_dir.exists()} ({len(list(kb_dir.glob('*')))} files)")
    print(f"  + Backups: {backup_dir.exists()} ({len(list(backup_dir.glob('*.zip')))} archives)")
except ImportError:
    print("[LOCAL SIMULATION] Running outside of Google Colab.")

## Step 2: Hardware Resource & GPU Memory Verification

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device: {device_name}")
    print(f"Total GPU Memory: {total_vram_gb:.2f} GB")
    
    if total_vram_gb < 12.0:
        print("[NOTICE] GPU memory is below 12 GB. 4-bit quantization (NF4) is strictly required.")
    else:
        print("[OK] GPU memory is sufficient for 4-bit Mistral-7B inference (~5.5 GB VRAM requirement).")
else:
    raise SystemError("GPU acceleration is not enabled. Go to Runtime -> Change runtime type -> Select T4 GPU.")

## Step 3: Install Required Inference Packages
Installs Hugging Face `transformers`, `accelerate`, `bitsandbytes` (for 4-bit memory efficiency), `fastapi`, `uvicorn`, and tunneling utilities.

In [ ]:
!pip install -q --upgrade transformers accelerate bitsandbytes fastapi uvicorn pydantic pyngrok httpx sentencepiece protobuf


## Step 4: Load Mistral-7B Model with 4-bit Quantization
Loads `mistralai/Mistral-7B-Instruct-v0.3` using 4-bit NF4 quantization to minimize GPU VRAM consumption to ~5.2 GB.

In [ ]:
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
HF_TOKEN = os.getenv("HF_TOKEN", None)  # Optional HF read token if accessing gated weights

# 4-Bit NF4 Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"[LOADING] Loading configured model: {MODEL_ID} in 4-bit NF4...")
t0 = time.time()

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        token=HF_TOKEN,
    )
    ACTIVE_MODEL = MODEL_ID
    print(f"[SUCCESS] {ACTIVE_MODEL} loaded successfully!")
except Exception as exc:
    print(f"\n[FATAL ERROR] Failed to load {MODEL_ID}: {exc}")
    print("[DIAGNOSTIC] Possible causes:")
    print("  1. Gated Access: Set HF_TOKEN in env or run `from huggingface_hub import login; login()'`")
    print("  2. Memory: Ensure T4 GPU is selected (Runtime -> Change runtime type -> T4 GPU)")
    print("[GOVERNANCE] Automatic model substitution is strictly disabled. Stopping execution.")
    raise

load_time = time.time() - t0
vram_allocated_gb = torch.cuda.memory_allocated() / (1024**3)
print(f"[SUCCESS] Model loaded in {load_time:.2f} seconds!")
print(f"[TELEMETRY] Current VRAM Allocated: {vram_allocated_gb:.2f} GB")


## Step 5: Test Model Generation Locally (Sanity & Latency Check)

In [ ]:
sample_messages = [
    {"role": "system", "content": "You are the ApexPrint Customer Support Assistant. Ground all answers strictly in evidence."},
    {"role": "user", "content": "What should I do if error E-101 paper jam occurs on the ApexPrint ProX-400? [Source: apexprint_troubleshooting.md, Page 1]"}
]

input_ids = tokenizer.apply_chat_template(sample_messages, tokenize=True, return_tensors="pt").to("cuda")

t0_gen = time.time()
with torch.no_grad():
    output_tokens = model.generate(
        input_ids,
        max_new_tokens=256,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
gen_latency = time.time() - t0_gen
new_tokens_count = len(output_tokens[0]) - len(input_ids[0])
tok_per_sec = new_tokens_count / gen_latency if gen_latency > 0 else 0

generated_text = tokenizer.decode(output_tokens[0][len(input_ids[0]):], skip_special_tokens=True)
print(f"=== GENERATED RESPONSE ({new_tokens_count} tokens in {gen_latency:.2f}s | {tok_per_sec:.1f} tok/s) ===")
print(generated_text)
print("=" * 60)

## Step 6: Define Authenticated OpenAI-Compatible REST Server
Creates a lightweight FastAPI app implementing `/v1/models` and `/v1/chat/completions` with Bearer token authentication.

In [ ]:
import os
import secrets
import time
from typing import List, Dict, Any, Optional
from fastapi import FastAPI, HTTPException, Security, Depends, status
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel

# Secure Runtime Authentication Token Generation
# Reads from environment or generates cryptographically secure token at runtime.
# Zero hardcoded tokens; never committed to version control.
INFERENCE_AUTH_TOKEN = os.getenv("INFERENCE_AUTH_TOKEN")
if not INFERENCE_AUTH_TOKEN:
    INFERENCE_AUTH_TOKEN = secrets.token_urlsafe(32)
    print("[SECURITY] Generated fresh 256-bit entropy session token.")
else:
    print("[SECURITY] Using INFERENCE_AUTH_TOKEN provided via environment.")

token_fingerprint = f"{INFERENCE_AUTH_TOKEN[:4]}...{INFERENCE_AUTH_TOKEN[-4:]}"
print(f"[SECURITY] Auth verification active (Fingerprint: {token_fingerprint})")

security = HTTPBearer(auto_error=False)

def verify_auth(credentials: Optional[HTTPAuthorizationCredentials] = Security(security)):
    if not credentials or credentials.credentials != INFERENCE_AUTH_TOKEN:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or missing Bearer authentication token",
            headers={"WWW-Authenticate": "Bearer"},
        )
    return credentials.credentials

app = FastAPI(title="ApexPrint Colab Remote Inference Server", version="1.0")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = None
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.2
    max_tokens: Optional[int] = 512

@app.get("/v1/models")
async def list_models():
    return {
        "data": [
            {"id": MODEL_ID, "object": "model", "owned_by": "mistralai"}
        ]
    }

@app.post("/v1/chat/completions", dependencies=[Depends(verify_auth)])
async def chat_completions(req: ChatCompletionRequest):
    msg_dicts = [{"role": m.role, "content": m.content} for m in req.messages]
    try:
        input_tokens = tokenizer.apply_chat_template(msg_dicts, tokenize=True, return_tensors="pt").to("cuda")
        t_start = time.time()
        with torch.no_grad():
            out_tokens = model.generate(
                input_tokens,
                max_new_tokens=min(req.max_tokens or 512, 1024),
                temperature=req.temperature or 0.2,
                do_sample=(req.temperature or 0.2) > 0.0,
                pad_token_id=tokenizer.pad_token_id,
            )
        latency = time.time() - t_start
        answer_text = tokenizer.decode(out_tokens[0][len(input_tokens[0]):], skip_special_tokens=True)
        
        return {
            "id": f"chatcmpl-{secrets.token_hex(6)}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": req.model or MODEL_ID,
            "choices": [
                {
                    "index": 0,
                    "message": {"role": "assistant", "content": answer_text.strip()},
                    "finish_reason": "stop",
                }
            ],
            "usage": {
                "prompt_tokens": len(input_tokens[0]),
                "completion_tokens": len(out_tokens[0]) - len(input_tokens[0]),
                "total_tokens": len(out_tokens[0]),
            },
            "latency_seconds": round(latency, 3),
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Inference runtime error: {str(e)}")


## Step 7: Start Background Uvicorn Server on Port 8001

In [ ]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8001, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2.0)
print("[OK] Uvicorn inference server listening locally on port 8001.")

## Step 8: Expose Public Tunnel (ngrok / Cloudflare Tunnel)
Generates a public URL to connect your local FastAPI backend / Streamlit UI to the Colab GPU.

In [ ]:
# Option A: ngrok Tunnel (Requires ngrok account & token)
from pyngrok import ngrok
# Optional: ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")
try:
    public_tunnel = ngrok.connect(8001)
    print("\n" + "=" * 60)
    print("PUBLIC INFERENCE ENDPOINT (ngrok):")
    print(f"COLAB_API_ENDPOINT={public_tunnel.public_url}/v1")
    print("COLAB_API_KEY=<Use the runtime session token from Step 6>")
    print("=" * 60)
    print("Configure the endpoint above in your local .env or Streamlit sidebar.")
    print("(Session token is never printed in tunnel logs to prevent exposure in screenshots).")
except Exception as e:
    print(f"[INFO] ngrok note: {e}")
    print("\n--- Option B: Cloudflare Tunnel (Zero signup required) ---")
    print("Run in a separate code cell:")
    print("!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb")
    print("!cloudflared tunnel --url http://127.0.0.1:8001")


## Step 9: End-to-End Self-Test via HTTP Client
Validates that authentication enforcement (HTTP 401 on missing token) and authorized inference (HTTP 200) work as expected.

In [ ]:
import httpx

# Test 1: Verify Unauthorized access is rejected (HTTP 401)
with httpx.Client() as client:
    resp_unauth = client.post("http://127.0.0.1:8001/v1/chat/completions", json={
        "messages": [{"role": "user", "content": "ping"}]
    })
    print(f"Auth Enforcement Check: HTTP {resp_unauth.status_code} (Expected: 401)")
    assert resp_unauth.status_code == 401, "Server must require Bearer token!"

# Test 2: Verify Authorized request succeeds (HTTP 200)
headers = {"Authorization": f"Bearer {INFERENCE_AUTH_TOKEN}"}
payload = {
    "model": MODEL_ID,
    "messages": [
        {"role": "system", "content": "You are a customer support assistant. Answer succinctly."},
        {"role": "user", "content": "What is the standard warranty on the ApexPrint ProX-400?"}
    ],
    "max_tokens": 128,
    "temperature": 0.1
}

with httpx.Client(timeout=30.0) as client:
    resp_auth = client.post("http://127.0.0.1:8001/v1/chat/completions", json=payload, headers=headers)
    print(f"Authorized Request Check: HTTP {resp_auth.status_code} (Expected: 200)")
    if resp_auth.status_code == 200:
        data = resp_auth.json()
        print("Generated text:", data["choices"][0]["message"]["content"])
        print("Latency:", data.get("latency_seconds"), "s")
        print("[SUCCESS] End-to-end self-test completed successfully!")

## Step 10: Safe Server Shutdown & GPU Memory Cleanup

In [ ]:
def shutdown_inference_service():
    """Disconnect tunnels, stop background server, and flush GPU memory."""
    print("[SHUTDOWN] Disconnecting tunnels...")
    try:
        ngrok.disconnect(public_tunnel.public_url)
        ngrok.kill()
    except Exception:
        pass
    
    print("[SHUTDOWN] Clearing CUDA VRAM cache...")
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print("[SHUTDOWN] Inference service shutdown cleanly.")

# Uncomment to execute shutdown:
# shutdown_inference_service()